# Data processing

The objective of this notebook is to prepare the dataset for subsequent analysis and modeling.

Based on the findings from the exploratory data analysis (EDA), several preprocessing steps are applied to improve data quality and consistency.

These transformations include handling missing values, correcting data types, removing unnecessary columns, and preparing variables for future analysis.

## Dataset

The dataset used in this notebook is the **TMDB Movie Metadata** dataset, available on Kaggle:

**TMDB Movie Metadata**
https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

It contains metadata for thousands of movies collected from **The Movie Database (TMDB)**, including titles, overviews, genres, keywords, cast, crew, ratings, release dates, and other useful information for content-based recommendation systems.

## Imports

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Darío\Desktop\OTROS\vector-recommender


## Load datasets

In [14]:
# 📍 Ensure we are working from the project root
project_root = Path.cwd()

# 📂 Define dataset paths (adjust if your structure differs)
movies_path = project_root / "data" / "raw" / "tmdb_5000_movie_dataset" / "tmdb_5000_movies.csv"
credits_path = project_root / "data" / "raw" / "tmdb_5000_movie_dataset" / "tmdb_5000_credits.csv"

# 📥 Load datasets
movies_df = pd.read_csv(movies_path)
credits_df = pd.read_csv(credits_path)

# 🔍 Quick inspection
print("Movies dataset shape:", movies_df.shape)
print("Credits dataset shape:", credits_df.shape)

display(movies_df.head())
display(credits_df.head())

Movies dataset shape: (4803, 20)
Credits dataset shape: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## Safe merge

In [15]:
# 📍 Ensure consistent key name for merging
credits_df = credits_df.rename(columns={"movie_id": "id"})

# 🔍 Sanity check: ensure titles match for the same id
title_mismatch = movies_df.merge(
    credits_df[["id", "title"]],
    on="id",
    how="inner",
    suffixes=("_movies", "_credits")
)

# 🚨 Validate that titles are consistent across datasets
inconsistent_titles = title_mismatch[
    title_mismatch["title_movies"] != title_mismatch["title_credits"]
]

print("Title mismatches found:", len(inconsistent_titles))

# 👀 Inspect mismatches if any exist
if len(inconsistent_titles) > 0:
    display(inconsistent_titles.head())

# 📦 Keep only relevant columns from credits
credits_reduced = credits_df[["id", "cast", "crew"]]

# 🔗 Final merge
merged_df = movies_df.merge(credits_reduced, on="id", how="inner")

# 📊 Validate final result
print("Merged dataset shape:", merged_df.shape)

# 👀 Preview
display(merged_df.head())

Title mismatches found: 0
Merged dataset shape: (4803, 22)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## Replace invalid zero values

In the TMDB dataset, zero values in teh `budget`, `revenue` and `runtime` columns typically indicate missing information rather than actual zero values. These entries are therefore replaced with `NaN` to correctly represent missing data.

In [16]:
merged_df["budget"] = merged_df["budget"].replace(0, np.nan)
merged_df["revenue"] = merged_df["revenue"].replace(0, np.nan)
merged_df["runtime"] = merged_df["runtime"].replace(0, np.nan)

## Extract Release Date Features

The `release_date` column is first converted to a datetime format to ensure consistency. Two additional temporal features are then extracted: the release year and the release month. These features can be useful for identifying temporal trends and may be valuable for subsequent analysis or modeling.

In [17]:
# Convert release_date to datetime format
merged_df["release_date"] = pd.to_datetime(
    merged_df["release_date"], errors="coerce"
)

# Extract temporal features
merged_df["release_year"] = merged_df["release_date"].dt.year
merged_df["release_month"] = merged_df["release_date"].dt.month

# Preview the new features
merged_df[["title", "release_date", "release_year", "release_month"]].head()

,title,release_date,release_year,release_month
0,Avatar,2009-12-10,2009.0,12.0
1,Pirates of the Caribbean: At World's End,2007-05-19,2007.0,5.0
2,Spectre,2015-10-26,2015.0,10.0
3,The Dark Knight Rises,2012-07-16,2012.0,7.0
4,John Carter,2012-03-07,2012.0,3.0


## Create Profit Feature

A new feature, `profit`, is created by subtracting the production budget from the total revenue. This variable provides an estimate of the financial return generated by each movie and can be useful for profitability analyses and predictive modeling.

In [18]:
# Calculate movie profit
merged_df["profit"] = merged_df["revenue"] - merged_df["budget"]

# Preview the new feature
merged_df[["title", "budget", "revenue", "profit"]].head()

,title,budget,revenue,profit
0,Avatar,237000000.0,2.787965e+09,2.550965e+09
1,Pirates of the Caribbean: At World's End,300000000.0,9.610000e+08,6.610000e+08
2,Spectre,245000000.0,8.806746e+08,6.356746e+08
3,The Dark Knight Rises,250000000.0,1.084939e+09,8.349391e+08
4,John Carter,260000000.0,2.841391e+08,2.413910e+07


## ⭐ Weighted Rating (IMDb-style ranking)

To build a more reliable movie rating system, we use a **weighted rating formula** similar to the one used by IMDb. This approach takes into account not only the average rating of a movie, but also the number of votes it has received, giving more importance to movies with a larger number of ratings.

#### 📌 Formula

$$
WR = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
$$

Where:

- **v** = number of votes (vote_count)  
- **R** = average rating of the movie (vote_average)  
- **C** = mean vote across the whole dataset  
- **m** = minimum number of votes required to be considered  

#### 🧠 Intuition

- Movies with **few votes** are pulled closer to the global average (C).  
- Movies with **many votes** rely more on their own rating (R).  
- This prevents highly rated but low-vote movies from ranking unfairly high.  

In [19]:
# Work on a copy to preserve the original dataframe
df = merged_df.copy()

# Calculate the global average rating (C)
C = df["vote_average"].mean()

# Define the minimum vote threshold (10th percentile)
m = df["vote_count"].quantile(0.10)

print(f"Minimum votes threshold (m): {m:.0f}")

# Select movies that meet the minimum vote threshold
qualified = df[df["vote_count"] >= m].copy()

# Calculate the weighted rating
v = qualified["vote_count"]
R = qualified["vote_average"]

qualified["weighted_rating"] = (
    (v / (v + m)) * R +
    (m / (v + m)) * C
)

# Merge the weighted rating back into the original dataframe
merged_df = merged_df.merge(
    qualified[["weighted_rating"]],
    left_index=True,
    right_index=True,
    how="left"
)

# Preview the new feature
merged_df[["title", "vote_average", "vote_count", "weighted_rating"]].head()

Minimum votes threshold (m): 12


,title,vote_average,vote_count,weighted_rating
0,Avatar,7.2,11800,7.198875
1,Pirates of the Caribbean: At World's End,6.9,4500,6.897852
2,Spectre,6.3,4466,6.299443
3,The Dark Knight Rises,7.6,9106,7.598016
4,John Carter,6.1,2124,6.099956


## Save Processed Dataset

After completing all preprocessing steps, the resulting dataset is saved for use in subsequent stages of the project, such as feature engineering, modeling, or further analysis.

In [20]:
# Save the processed dataset
merged_df.to_csv("data/processed/tmdb_5000_movie_dataset/tmdb_5000_processed.csv", index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
